In [ ]:
import re
import requests
import numpy as np
import pandas as pd

# Define functions

In [ ]:
def query_opensky_network(
    lat_min: float | None = None,
    lat_max: float | None = None,
    lon_min: float | None = None,
    lon_max: float | None = None,
) -> pd.DataFrame:
    if lat_min is None:
        response = requests.get("https://opensky-network.org/api/states/all")
    else:
        response = requests.get(
            "https://opensky-network.org/api/states/all",
            params={
                "lamin": lat_min,
                "lomin": lon_min,
                "lamax": lat_max,
                "lomax": lon_max,
            },
        )

    cols = [
        "icao24",
        "callsign",
        "origin_country",
        "unix_time_position [s]",
        "unix_last_contact [s]",
        "longitude [WGS84 °]",
        "latitude [WGS84 °]",
        "baro_altitude [m]",
        "is_on_ground",
        "velocity_over_ground [m/s]",
        "true_track [°]",
        "vertical_rate [m/s]",
        "sensors",
        "geo_altitude [m]",
        "squawk",
        "spi",
        "category",
    ]

    return pd.DataFrame(response.json()["states"], columns=cols)


def load_ADSC_file(path: str) -> pd.DataFrame:
    """
    Load the ADS-C Air Traffic Dataset. https://doi.org/10.5281/zenodo.10041840
    """
    waypoints = []

    # Load file.
    with open(path, "r") as file:
        text = file.read()

    # Identify registrations.
    registrations = text.split("\n\n")

    for registration in registrations:
        if len(registration) == 0:
            continue

        # Parse registration.
        lines = registration.split("\n")
        header = lines[0:3]
        crc = lines[-1]
        tags = "\n".join(lines[3:-1])

        icao_string = [h for h in header[0].split("    ") if h.startswith("ICAO ID: ")]
        assert len(icao_string) == 1
        icao = icao_string[0].replace("ICAO ID: ", "")

        tags = re.findall(r"^(.*?)\n((?:[ \t].*\n?)*)", tags, flags=re.MULTILINE)
        # We only care about waypoints, i. e. Tag 07.
        tags = ["\n".join(tag) for tag in tags if tag[0].startswith("Tag 07")]

        # Parse the tags.
        for tag in tags:
            lat_string = [
                h for h in tag.split("\n") if h.strip().startswith("Latitude: ")
            ]
            assert len(lat_string) == 1
            lat = lat_string[0].strip().replace("Latitude: ", "")

            lon_string = [
                h for h in tag.split("\n") if h.strip().startswith("Longitude: ")
            ]
            assert len(lon_string) == 1
            lon = lon_string[0].strip().replace("Longitude: ", "")

            alt_string = [
                h for h in tag.split("\n") if h.strip().startswith("Altitude: ")
            ]
            assert len(alt_string) == 1
            alt = alt_string[0].strip().replace("Altitude: ", "").replace("ft", "")

            timestamp_string = [
                h for h in tag.split("\n") if h.strip().startswith("Timestamp: ")
            ]
            assert len(timestamp_string) == 1
            timestamp = timestamp_string[0].strip().replace("Timestamp: ", "")

            waypoints.append(
                {
                    "icao": icao,
                    "lat [deg]": float(lat),
                    "lon [deg]": float(lon),
                    "alt [ft]": float(alt),
                    "datetime": timestamp,
                }
            )

    df = pd.DataFrame(waypoints)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["icao"] = df["icao"].astype("str")
    df.set_index(["icao", "datetime"], inplace=True)
    df.sort_index(inplace=True)

    return df


def load_opensky_data_samples(path: str) -> pd.DataFrame:
    # Doc: https://s3.opensky-network.org/data-samples/states/README.txt
    df = pd.read_csv(path)
    df.query("~`onground`", inplace=True)
    df.drop(
        columns=["alert", "spi", "squawk", "lastposupdate", "lastcontact", "onground"],
        inplace=True,
    )
    df["time"] = pd.to_datetime(df["time"], unit="s")
    df.dropna(how="any", inplace=True)
    df.set_index(["icao24", "time"], inplace=True)
    df.sort_index(inplace=True)

    df["vlon"] = np.sin(np.radians(df["heading"])) * df["velocity"]
    df["vlat"] = -np.cos(np.radians(df["heading"])) * df["velocity"]
    df.rename(columns={"vertrate": "vz", "geoaltitude": "alt"}, inplace=True)
    df["callsign"] = df["callsign"].str.strip()
    df["cross_section"] = np.nan
    df = df[["callsign", "lat", "lon", "alt", "vlat", "vlon", "vz", "cross_section"]]

    return df

# Load

In [ ]:
lat_min = 45.351944
lon_min = 5.449722
lat_max = 48.189320
lon_max = 10.890690

In [ ]:
# Extracted CSV file from:
# https://s3.opensky-network.org/data-samples/states/2022-06-27/23/states_2022-06-27-23.csv.tar
path = "/home/user/Downloads/states_2022-06-27-23.csv/states_2022-06-27-23.csv"
path_aircraft_types = "/home/user/Downloads/aircraft-database-complete-2025-08.csv"
df = load_opensky_data_samples(path)
df = df.query("(@lat_min <= `lat` <= @lat_max) and (@lon_min <= `lon` <= @lon_max)")

In [ ]:
df_aircrafts = pd.read_csv(path_aircraft_types, usecols=np.arange(20))
df_aircrafts.columns = [c.replace("'", "") for c in df_aircrafts.columns]
df_aircrafts["icao24"] = df_aircrafts["icao24"].str.replace("'", "")
df_aircrafts.set_index("icao24", inplace=True)
df_aircrafts_relevant = df_aircrafts.loc[df.reset_index()["icao24"].unique(), ["manufacturerIcao", "model"]]

In [ ]:
df = df.join(df_aircrafts_relevant)
df = df[["lat", "lon", "alt", "vlat", "vlon", "vz", "cross_section", "callsign", "manufacturerIcao", "model"]]
df.loc[:, "cross_section"] = 1.

In [ ]:
df

# Save

In [ ]:
df.to_csv("data_opensky_2022-06-27.csv")

# Plot

In [ ]:
import folium

highlight = "RYR3RR"

map = folium.Map(
    location=((lat_max + lat_min) / 2, (lon_max + lon_min) / 2),
    zoom_start=8,
)
for sign, track in df.groupby("callsign"):
    folium.PolyLine(
        [(p["lat"], p["lon"]) for _, p in track.iterrows()],
        tooltip=f"Callsign: {sign}<br />Manufacturer: {track['manufacturerIcao'].iloc[0]}<br />Model: {track['model'].iloc[0]}",
        color="#DC3312" if sign == highlight else "#3366CC",
    ).add_to(map)

folium.LatLngPopup().add_to(map)

map.save("map.html")